## "Quiz Master"

This is my idea for a "visible agent loop":

- The user tells the system on what topic they wants questions to be asked and also the number of questions
- The agent first prepares a list of questions behind the scenes
- The system asks the questions one by one - lets make it multiple choice questions with one correct answer
- As soon as the user answers them, the system prints a status - a green tick for correct answer, a red cross for wrong ones and a dash for remaining questions

In [1]:
import os
from dotenv import load_dotenv
from rich.console import Console
from pydantic import BaseModel, Field
from agents import Agent, Runner, function_tool
from agents import set_tracing_export_api_key
import gradio as gr


In [2]:
load_dotenv(override=True)
MODEL_NAME = "gpt-5.4-mini"
NUM_QUESTIONS = 10

In [3]:
os.environ["OPENAI_LOG"] = "debug"
set_tracing_export_api_key(os.getenv("OPENAI_TRACING_KEY"))

In [4]:
class Choice(BaseModel):
    optionId: str = Field(..., description="The unique identifier for the choice option. Foe example, 'A', 'B', 'C', etc.")
    text: str = Field(..., description="The text of the choice option.")

class QuestionAndAnswer(BaseModel):
    question: str = Field(..., description="The question to be answered.")
    choices: list[Choice] = Field(..., description="The list of choice options for the question.")
    answer: str = Field(..., description="The correct answer to the question, represented by the optionId of the correct choice.")

class QuestionsAndAnswers(BaseModel):
    questions: list[QuestionAndAnswer] = Field(..., description="A list of questions and their corresponding answers.")

In [5]:
INSTRUCTIONS = """
You are a questions generator. Generate a set of multiple-choice questions with four options each. Each question should have one correct answer. 
The questions should be clear and concise, and the options should be plausible to make the quiz challenging.
"""
questions_generator_agent = Agent(
    name="QuestionsGenerator",
    instructions=INSTRUCTIONS,
    output_type=QuestionsAndAnswers,
    model=MODEL_NAME
    )

In [6]:
# task = f"Generate {NUM_QUESTIONS} multiple-choice questions on the topic of Python programming."
# result = await Runner.run(questions_generator_agent, task)

In [7]:
# question_answers = result.final_output
# print(question_answers)

In [8]:
# 0 for wrong answer, 1 for correct answer
answers = [None] * NUM_QUESTIONS

In [9]:
# def check_answer(question: QuestionAndAnswer, user_answer: str) -> bool:
#     """
#     Check if the user's answer is correct for the given question.

#     Args:
#         question (QuestionAndAnswer): The question to check against.
#         user_answer (str): The user's answer, represented by the optionId.

#     Returns:
#         bool: True if the answer is correct, False otherwise.
#     """
    
#     return question.answer == user_answer

In [10]:
# # generate questions
# questions = [
#     {"question": "What is the capital of France?", "answer": "Paris"},
#     {"question": "Who wrote 'Romeo and Juliet'?", "answer": "William Shakespeare"},
#     {"question": "What is the largest planet in our solar system?", "answer": "Jupiter"}
# ]


In [11]:
@function_tool
def record_answer(question_index: int, is_correct: bool) -> None:
    """
    Record the user's answer for a specific question.

    Args:
        question_index (int): The index of the question in the questions list.
        is_correct (bool): Whether the user's answer is correct.
    """
    global answers
    if 0 <= question_index < NUM_QUESTIONS:
        answers[question_index] = is_correct
    else:
        print(f"Invalid question index: {question_index}")

In [12]:
@function_tool
def print_status() -> str:
    """
    Returns a string representing the current status of the quiz, showing which questions have been answered correctly, incorrectly, or not answered yet.
    """
    global answers
    result = ""
    for index in range(NUM_QUESTIONS):
        if index >= len(answers):
            result += "[bold yellow]-[/bold yellow] "
        elif answers[index] == True:
            result += "[bold green]✓[/bold green] "
        elif answers[index] == False:
            result += "[bold red]✗[/bold red] "
        else:
            result += "[bold yellow]-[/bold yellow] "
    return result

In [13]:
questions_generator_agent_tool = questions_generator_agent.as_tool(tool_name="questions_generator", tool_description="Generates multiple-choice questions with four options each.")

In [ ]:
INSTRUCTIONS = f"""
You are a quiz master bot.

The first message from the user would be the topic on which they want to be quizzed.

Then use the questions_generator tool to generate {NUM_QUESTIONS} multiple-choice questions with four options each on that topic. Each question should have one correct answer.

Your task is to present the questions to the user one by one. 

The questions will be provided in a structured format, and you should ensure that the user understands how to respond with their chosen option (e.g., 'A', 'B', 'C', or 'D').

As soon as the user provides the answer, use the record_answer tool to save the answer, by passing the question index and whether the answer is correct (True/False).

And then call print_status tool to get progress as a string. Display this string after each answer they submit. 

If the user provides an invalid answer, prompt them to provide a valid answer.

"""

quiz_master_bot = Agent(name="QuizMasterBot", instructions=INSTRUCTIONS, model=MODEL_NAME, tools=[questions_generator_agent_tool, record_answer, print_status])

In [15]:
last_response = None

In [16]:
async def async_chat(message, history):
    global last_response
    if last_response is None:
        # first turn - noting to carry forward
        agent_input = message
    else:
        # subsequent turns - carry forward the last response
        agent_input = last_response.to_input_list() + [{"role": "user", "content": message}]

    last_response = await Runner.run(quiz_master_bot, agent_input)
    return last_response.final_output
    

In [ ]:
gr.ChatInterface(
    async_chat,
    title="Quiz Master",
    description="Hi! I am a quiz master. On what topic can I ask you questions today?").launch(inbrowser=True)


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


/Users/arun/Development/udemy/agents/.venv/lib/python3.12/site-packages/gradio/routes.py:1541: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
/Users/arun/Development/udemy/agents/.venv/lib/python3.12/site-packages/gradio/routes.py:1541: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
/Users/arun/Development/udemy/agents/.venv/lib/python3.12/site-packages/gradio/routes.py:1541: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
/Users/arun/Development/udemy/agents/.venv/lib/python3.12/site-packages/gradio/routes.py:1541: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 